# GeoVision-CLIP Cali — Exploracion de tiles

Entender la estructura, distribucion y calidad de los 5,000 tiles de entrenamiento.

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
INPUT = Path("/kaggle/input")
TILES_PATH = INPUT / "datasets/edwardsx/geovision-tiles-sit2"
if not TILES_PATH.exists():
    TILES_PATH = INPUT / "geovision-tiles-sit2"

OUTPUT = Path("/kaggle/working")
FIG_DIR = OUTPUT / "figuras_tiles"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASES = [
    "contaminacion_alta_NO2",
    "contaminacion_alta_SO2",
    "ozono_anomalo",
    "vegetacion_densa",
    "suelo_urbano",
]

COLORES = {
    "contaminacion_alta_NO2": "#e74c3c",
    "contaminacion_alta_SO2": "#e67e22",
    "ozono_anomalo": "#f1c40f",
    "vegetacion_densa": "#27ae60",
    "suelo_urbano": "#7f8c8d",
}

BANDAS_OPTICAS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]
TILE_PX = 64

COLS_ERA5 = ["era5_T2m", "era5_Td2m", "era5_u10", "era5_v10", "era5_BLH", "era5_RH850", "era5_psurf", "era5_precip"]
COLS_MODIS = ["modis_AOD_047", "modis_AOD_055", "modis_WV"]
COLS_S5P = ["no2", "so2", "o3"]

## Cargar datos

In [ ]:
npz = np.load(TILES_PATH / "tiles_train.npz", allow_pickle=False)
tiles = npz["data"]
bands = list(npz["bands"])
meta = pd.read_parquet(TILES_PATH / "tiles_meta.parquet")

print(f"Tiles shape: {tiles.shape}")
print(f"Bandas ({len(bands)}): {bands}")
print(f"Meta shape: {meta.shape}")
print(f"Meta columnas ({len(meta.columns)}): {list(meta.columns)}")

In [ ]:
print("Valores del npz:")
print(f"  dtype: {tiles.dtype}")
print(f"  rango: {tiles.min():.2f} a {tiles.max():.2f}")
print(f"  NaN: {np.isnan(tiles).sum()}")
print(f"  ceros: {(tiles == 0).mean()*100:.2f}%")
print()
print("Distribucion por clase:")
print(meta["clase"].value_counts().to_string())

## Distribucion espacial

In [ ]:
print(f"Rango lat: {meta['lat'].min():.4f} a {meta['lat'].max():.4f}")
print(f"Rango lon: {meta['lon'].min():.4f} a {meta['lon'].max():.4f}")
print()
print("Media lat/lon por clase:")
print(meta.groupby("clase")[["lat", "lon"]].agg(["mean", "count", "std"]).round(4).to_string())

In [ ]:
PANEL_PATH = INPUT / "datasets/juanjoseorozcolopez/geovision-fuentes"
if not PANEL_PATH.exists():
    PANEL_PATH = INPUT / "geovision-fuentes"
estaciones = pd.read_csv(PANEL_PATH / "dagma" / "estaciones_metadata.csv")
print("Estaciones DAGMA:")
for _, row in estaciones.iterrows():
    print(f"  {row['nombre_est']:30s} ({row['nombre_fgda']:4s})  lat={row['latitud']:.4f}  lon={row['longitud']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
for clase in CLASES:
    mask = meta["clase"] == clase
    ax.scatter(meta.loc[mask, "lon"], meta.loc[mask, "lat"],
               c=COLORES[clase], label=clase[:25], alpha=0.4, s=8)
for _, row in estaciones.iterrows():
    ax.scatter(row["longitud"], row["latitud"], c="black", s=80, marker="D", zorder=5)
    ax.annotate(row["nombre_est"], (row["longitud"], row["latitud"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8, zorder=6)
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title("Tiles por clase + estaciones DAGMA")
ax.legend(markerscale=3, loc="upper left")
fig.savefig(str(FIG_DIR / "mapa_tiles_estaciones.png"), dpi=150, bbox_inches="tight")
print(f"Mapa: {FIG_DIR / 'mapa_tiles_estaciones.png'}")

## Separacion de clases

In [ ]:
print("Estadisticas NDVI y NDBI por clase:")
for clase in CLASES:
    mask = meta["clase"] == clase
    print(f'{clase[:30]:30s} | NDVI: {meta.loc[mask, "ndvi"].mean():.3f} +/- {meta.loc[mask, "ndvi"].std():.3f} | NDBI: {meta.loc[mask, "ndbi"].mean():.3f} +/- {meta.loc[mask, "ndbi"].std():.3f}')
print()
print("Distancia entre centros (NDVI, NDBI):")
centros = meta.groupby("clase")[["ndvi", "ndbi"]].mean().values
for i, c1 in enumerate(CLASES):
    for j, c2 in enumerate(CLASES):
        if i < j:
            d = np.linalg.norm(centros[i] - centros[j])
            print(f"  {c1[:20]:20s} vs {c2[:20]:20s}: {d:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for clase in CLASES:
    mask = meta["clase"] == clase
    ax.scatter(meta.loc[mask, "ndvi"], meta.loc[mask, "ndbi"],
               c=COLORES[clase], label=clase[:25], alpha=0.3, s=5)
ax.set_xlabel("NDVI"); ax.set_ylabel("NDBI")
ax.set_title("Separacion de clases en espacio NDVI vs NDBI")
ax.legend(markerscale=5)
fig.savefig(str(FIG_DIR / "ndvi_ndbi_scatter.png"), dpi=130, bbox_inches="tight")
print(f"Scatter: {FIG_DIR / 'ndvi_ndbi_scatter.png'}")


## Distribucion temporal

In [ ]:
meta["fecha_dt"] = pd.to_datetime(
    meta["time_s2"].astype(str).str.split("_").str[0],
    format="%Y%m%dT%H%M%S"
)
meta["ano"] = meta["fecha_dt"].dt.year
meta["mes"] = meta["fecha_dt"].dt.month

print("Tiles por ano y clase:")
print(pd.crosstab(meta["clase"], meta["ano"]).to_string())
print()
print("Tiles por mes calendario:")
print(pd.crosstab(meta["clase"], meta["mes"]).to_string())
print()
print("Fechas unicas por clase:")
print(meta.groupby("clase")["fecha_dt"].nunique().to_string())
print()
print("Rango temporal por clase:")
print(meta.groupby("clase")["fecha_dt"].agg(["min", "max"]).to_string())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
crosstab_ano = pd.crosstab(meta["clase"], meta["ano"])
crosstab_ano.T.plot(kind="bar", ax=axes[0], color=COLORES.values())
axes[0].set_title("Tiles por ano y clase")
axes[0].set_ylabel("Cantidad de tiles")
axes[0].legend(labels=[c[:20] for c in CLASES])

crosstab_mes = pd.crosstab(meta["clase"], meta["mes"])
crosstab_mes.T.plot(kind="bar", ax=axes[1], color=COLORES.values())
axes[1].set_title("Tiles por mes calendario y clase")
axes[1].set_ylabel("Cantidad de tiles")
axes[1].legend(labels=[c[:20] for c in CLASES])
fig.savefig(str(FIG_DIR / "tiles_temporal.png"), dpi=130, bbox_inches="tight")
print(f"Grafico: {FIG_DIR / 'tiles_temporal.png'}")


## Cobertura MODIS

In [ ]:
print("Cobertura MODIS por clase:")
for c in COLS_MODIS:
    print(f"\n{c}:")
    for clase in CLASES:
        mask = meta["clase"] == clase
        total = mask.sum()
        validos = meta.loc[mask, c].notna().sum()
        media = meta.loc[mask, c].mean()
        print(f"  {clase[:30]:30s} {validos:>4d}/{total} ({validos/total*100:5.1f}%) media={media:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, c in enumerate(COLS_MODIS):
    for clase in CLASES:
        mask = meta["clase"] == clase
        vals = meta.loc[mask, c].dropna()
        axes[i].hist(vals, bins=30, alpha=0.4, label=clase[:20], color=COLORES[clase])
    axes[i].set_title(c)
    axes[i].set_xlabel("Valor")
    axes[i].legend(fontsize=7)
fig.suptitle("Distribucion de valores MODIS por clase")
fig.savefig(str(FIG_DIR / "modis_cobertura.png"), dpi=130, bbox_inches="tight")
print(f"Grafico: {FIG_DIR / 'modis_cobertura.png'}")


## Pseudo-labels S5P

In [ ]:
print("Estadisticas de pseudo-labels S5P por clase:")
for col in COLS_S5P:
    print(f"\n{col} (mol/m2):")
    for clase in CLASES:
        vals = meta.loc[meta["clase"] == clase, col].dropna()
        if len(vals) > 0:
            print(f"  {clase[:30]:30s} n={len(vals):>4d}  media={vals.mean():.6f}  p50={vals.median():.6f}  p90={vals.quantile(0.9):.6f}  p99={vals.quantile(0.99):.6f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mapa_col_clase = {"no2": "contaminacion_alta_NO2", "so2": "contaminacion_alta_SO2", "o3": "ozono_anomalo"}
for i, col in enumerate(COLS_S5P):
    clase = mapa_col_clase[col]
    vals = meta.loc[meta["clase"] == clase, col]
    axes[i].hist(vals, bins=30, alpha=0.7, color=COLORES[clase], label=clase[:20])
    axes[i].axvline(vals.median(), color="black", ls="--", label=f"p50={vals.median():.6f}")
    axes[i].set_title(f"{col} - solo clase {clase[:15]}")
    axes[i].set_xlabel("mol/m2")
    axes[i].legend(fontsize=8)
fig.suptitle("Distribucion de pseudo-labels S5P")
fig.savefig(str(FIG_DIR / "s5p_pseudolabels.png"), dpi=130, bbox_inches="tight")
print(f"Grafico: {FIG_DIR / 's5p_pseudolabels.png'}")